### Ingestion bronze — raw → table Delta gouvernée

**Rôle de ce notebook, et seulement celui-ci :**
- Lire le CSV le plus récent déposé par ADF dans `bronze/<table>/ingestion_date=.../`
- Appliquer le schéma explicite (aucune transformation métier)
- Ajouter des colonnes de traçabilité
- Écrire comme table Delta, enregistrée dans Unity Catalog

**Ce que ce notebook ne fait JAMAIS :** corriger la casse, filtrer des lignes invalides, joindre des tables entre elles. Ça reste le travail exclusif de silver — bronze doit rester un miroir fidèle et interrogeable de la source, rien de plus.

In [0]:
%run "../libs/schema_definitions"

`%run` exécute l'autre notebook comme s'il faisait partie de celui-ci — toutes ses variables et fonctions (`get_schema`, `schema_ventes`...) deviennent disponibles ici, comme un `import` Python classique mais au niveau notebook. Le chemin est relatif : `..` remonte d'un dossier (`01_bronze` → `Projet_Ventes`), puis redescend dans `libs`.

In [0]:
from pyspark.sql import functions as F

In [0]:
# MAGIC Pourquoi ces 3 colonnes de traçabilité
# MAGIC 

# MAGIC - `_ingestion_date` : la date du lot ADF traité (extraite du nom du dossier).
# MAGIC   Sert aussi de **clé de partitionnement** physique de la table Delta —
# MAGIC   voir plus bas pourquoi c'est ce qui rend ce notebook rejouable sans risque.
# MAGIC - `_ingested_at` : horodatage exact de l'exécution *de ce notebook*
# MAGIC   (différent de `_ingestion_date` : on peut très bien traiter le 8 juillet
# MAGIC   un lot déposé le 7, en cas de retard). Utile pour debug un pipeline lent.
# MAGIC - `_source_file` : chemin complet du fichier physique d'origine. Si un jour
# MAGIC   une valeur suspecte apparaît en gold, on peut remonter jusqu'au fichier
# MAGIC   CSV exact qui l'a introduite — c'est la garantie d'audit qu'on avait
# MAGIC   évoquée dès la conception de la zone `raw`.

def get_latest_ingestion_folder(table_name: str) -> str:
    """
    Repère le dossier ingestion_date=YYYYMMDD le plus récent pour une source
    donnée. Le tri alphabétique suffit ici car le format YYYYMMDD trie
    correctement dans l'ordre chronologique (contrairement à DD/MM/YYYY).
    """
    base_path = f"abfss://bronze@adlsjopadev.dfs.core.windows.net/{table_name}/"
    entries = dbutils.fs.ls(base_path)

    ingestion_folders = [
        e.name.rstrip("/") for e in entries
        if e.name.startswith("ingestion_date=")
    ]

    if not ingestion_folders:
        raise FileNotFoundError(
            f"Aucun dossier ingestion_date trouvé sous {base_path}. "
            f"Le pipeline ADF a-t-il bien tourné ?"
        )

    latest = sorted(ingestion_folders)[-1]
    return latest

In [0]:
def ingest_source(table_name: str) -> None:
    """
    Ingère la dernière livraison raw→bronze d'une source donnée vers sa
    table Delta bronze, avec traçabilité et idempotence par partition.
    
    Idempotence obtenue via replaceWhere plutôt qu'un réglage de session :
    sur un compute gouverné par Unity Catalog (Serverless notamment),
    la modification de certaines configs Spark globales est restreinte
    par design, pour isoler les sessions des différents utilisateurs.
    """
    latest_folder = get_latest_ingestion_folder(table_name)
    ingestion_date = latest_folder.split("=")[1]
    file_path = f"abfss://bronze@adlsjopadev.dfs.core.windows.net/{table_name}/{latest_folder}/"

    df = spark.read.csv(file_path, header=True, schema=get_schema(table_name))

    df_enriched = (
        df
        .withColumn("_ingestion_date", F.lit(ingestion_date))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    target_table = f"bronze.ventes.{table_name}"
    table_exists = spark.catalog.tableExists(target_table)

    writer = df_enriched.write.format("delta")

    if not table_exists:
        # Premher run : on crée la table, partitionnée par date d'ingestion
        writer.mode("overwrite").partitionBy("_ingestion_date").saveAsTable(target_table)
    else:
        # Runs suivants : on ne remplace QUE la partition du jour traité,
        # les autres dates déjà en table restent intactes
        (
            writer
            .mode("overwrite")
            .option("replaceWhere", f"_ingestion_date = '{ingestion_date}'")
            .saveAsTable(target_table)
        )

    n_rows = df_enriched.count()
    print(f"[{table_name}] {n_rows} lignes ingérées → {target_table} "
          f"(partition _ingestion_date={ingestion_date})")

In [0]:
def ingest_source(table_name: str) -> None:
    """
    Ingère la dernière livraison raw→bronze d'une source donnée vers sa
    table Delta bronze, avec traçabilité et idempotence par partition.
    """
    latest_folder = get_latest_ingestion_folder(table_name)
    ingestion_date = latest_folder.split("=")[1]
    file_path = f"abfss://bronze@adlsjopadev.dfs.core.windows.net/{table_name}/{latest_folder}/"

    df = spark.read.csv(file_path, header=True, schema=get_schema(table_name))

    df_enriched = (
        df
        .withColumn("_ingestion_date", F.lit(ingestion_date))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    target_table = f"bronze.ventes.{table_name}"

    (
        df_enriched.write
        .format("delta")
        .mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("_ingestion_date")
        .saveAsTable(target_table)
    )

    n_rows = df_enriched.count()
    print(f"[{table_name}] {n_rows} lignes ingérées → {target_table} "
          f"(partition _ingestion_date={ingestion_date})")

In [0]:
# MAGIC Même remarque que pour le pipeline ADF : on boucle sur une liste plutôt
# MAGIC que de dupliquer le code deux fois. Si une 3ème source arrive un jour,
# MAGIC une seule ligne à ajouter ici.

for table in ["ventes", "retours"]:
    ingest_source(table)

In [0]:
%sql
SELECT * FROM bronze.ventes.ventes LIMIT 5;

In [0]:
%sql
SELECT * FROM bronze.ventes.retours LIMIT 5;